# Блок 1. Практика: Docker и Docker Compose

В этом notebook мы познакомимся с основами Docker: научимся запускать контейнеры, работать с томами и собирать собственные образы.

Команды Docker выполняются в терминале. Откройте терминал и следуйте инструкциям.

## Часть 1: Основы Docker

### 1.1 Запуск первого контейнера

Начнём с простого — запустим PostgreSQL. Без Docker вам пришлось бы скачивать дистрибутив, устанавливать, настраивать. С Docker это одна команда:

```bash
docker run -d --name my-postgres -e POSTGRES_PASSWORD=secret postgres:16
```

Разберём параметры:
- `-d` — запуск в фоновом режиме (detached)
- `--name my-postgres` — имя контейнера
- `-e POSTGRES_PASSWORD=secret` — переменная окружения
- `postgres:16` — образ и тег (версия)

### 1.2 Управление контейнерами

```bash
# Список запущенных контейнеров
docker ps

# Список всех контейнеров (включая остановленные)
docker ps -a

# Логи контейнера
docker logs my-postgres

# Следить за логами в реальном времени
docker logs -f my-postgres

# Подключиться к контейнеру (запустить bash внутри)
docker exec -it my-postgres bash

# Выполнить команду внутри контейнера
docker exec my-postgres psql -U postgres -c "SELECT version();"
```

### 1.3 Проблема: данные теряются

Остановите и удалите контейнер:

```bash
docker stop my-postgres
docker rm my-postgres
```

Все данные, которые были в базе, исчезли. Для production это катастрофа. Решение — тома (volumes).

### 1.4 Тома (volumes)

Том — это способ сохранить данные вне контейнера. При удалении контейнера том остаётся.

```bash
# Запуск с именованным томом
docker run -d \
  --name my-postgres \
  -e POSTGRES_PASSWORD=secret \
  -v pgdata:/var/lib/postgresql/data \
  postgres:16
```

Теперь данные сохраняются в томе `pgdata`. Можете удалить контейнер и создать новый с тем же томом — данные останутся.

```bash
# Список томов
docker volume ls

# Информация о томе
docker volume inspect pgdata

# Удаление тома (осторожно — данные будут потеряны!)
docker volume rm pgdata
```

### 1.5 Проброс портов

По умолчанию контейнер изолирован от хост-системы. Чтобы подключиться к PostgreSQL с хоста, нужно пробросить порт:

```bash
docker run -d \
  --name my-postgres \
  -e POSTGRES_PASSWORD=secret \
  -v pgdata:/var/lib/postgresql/data \
  -p 5432:5432 \
  postgres:16
```

Формат: `-p ХОСТ:КОНТЕЙНЕР`. Теперь PostgreSQL доступен на `localhost:5432`.

### 1.6 Проверка подключения из Python

Убедитесь, что контейнер запущен с пробросом порта, затем выполните ячейку ниже.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="postgres",
    user="postgres",
    password="secret"
)

cursor = conn.cursor()
cursor.execute("SELECT version();")
result = cursor.fetchone()
if result:
    print(result[0])

cursor.close()
conn.close()

### 1.7 Очистка

Перед переходом к следующей части остановите контейнер:

```bash
docker stop my-postgres
docker rm my-postgres
```

## Часть 2: Dockerfile

Dockerfile — это инструкция для сборки образа. Каждая команда создаёт новый слой, который кэшируется.

### 2.1 Простой пример

Создадим простое Python-приложение и упакуем его в Docker.

Создайте файл `hello.py`:

```python
import time
import os

name = os.getenv("NAME", "World")

while True:
    print(f"Hello, {name}! The time is {time.strftime('%H:%M:%S')}")
    time.sleep(5)
```

Создайте `Dockerfile`:

```dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY hello.py .

CMD ["python", "hello.py"]
```

### 2.2 Сборка и запуск

```bash
# Сборка образа
docker build -t hello-app .

# Запуск контейнера
docker run --name hello hello-app

# Ctrl+C для остановки, затем:
docker rm hello

# Запуск с переменной окружения
docker run --name hello -e NAME="Docker" hello-app
```

### 2.3 Слои и кэширование

Каждая инструкция в Dockerfile создаёт слой. Docker кэширует слои и переиспользует их при повторной сборке.

Попробуйте изменить `hello.py` и пересобрать образ:

```bash
docker build -t hello-app .
```

Обратите внимание: слои до `COPY hello.py` взяты из кэша, пересобирается только изменившийся слой.

**Важно:** порядок инструкций влияет на эффективность кэширования. Сначала копируйте то, что меняется редко (зависимости), потом — то, что меняется часто (код).

### 2.4 Dockerfile с uv

Для реальных проектов с зависимостями используем uv — быстрый менеджер пакетов Python.

```dockerfile
FROM debian:bookworm-slim

# Устанавливаем uv из официального образа
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

WORKDIR /app

# Сначала копируем файлы зависимостей (для кэширования)
COPY pyproject.toml uv.lock ./

# Устанавливаем зависимости
RUN uv sync --frozen

# Копируем код приложения
COPY src/ ./src/

# Запуск через uv
CMD ["uv", "run", "python", "src/main.py"]
```

Преимущества:
- `debian:bookworm-slim` меньше, чем `python:3.11`
- uv устанавливает зависимости очень быстро
- Слой с зависимостями кэшируется отдельно от кода

## Часть 3: Подготовка к заданию

В директории `app/` находится готовое приложение `log_generator.py`. Изучим его перед тем, как упаковывать в Docker.

### 3.1 Структура генератора

Генератор делает следующее:

1. Подключается к PostgreSQL
2. Создаёт таблицу `auth_events` (если её нет)
3. В бесконечном цикле генерирует события аутентификации
4. Имитирует как нормальную активность, так и атаки

Конфигурация через переменные окружения:
- `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` — подключение к БД
- `EVENTS_PER_BATCH` — событий за раз (по умолчанию 10)
- `BATCH_INTERVAL` — интервал в секундах (по умолчанию 5)
- `ATTACK_PROBABILITY` — вероятность атаки (по умолчанию 0.1)

### 3.2 Зависимости

Генератор использует `psycopg2-binary` для работы с PostgreSQL. Зависимость уже указана в корневом `pyproject.toml`.

Для вашего Dockerfile понадобится:
1. Скопировать `pyproject.toml` и `uv.lock` из корня репозитория
2. Установить зависимости через `uv sync`
3. Скопировать `lesson01/app/log_generator.py`

### 3.3 Тестовый запуск (без Docker)

Сначала запустите PostgreSQL:

```bash
docker run -d \
  --name test-postgres \
  -e POSTGRES_USER=analyst \
  -e POSTGRES_PASSWORD=security123 \
  -e POSTGRES_DB=security_logs \
  -p 5432:5432 \
  postgres:16
```

Затем запустите генератор локально:

```bash
cd lesson01/app
DB_HOST=localhost DB_USER=analyst DB_PASSWORD=security123 DB_NAME=security_logs uv run python log_generator.py
```

Вы должны увидеть логи генерации событий. Ctrl+C для остановки.

### 3.4 Проверка данных

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="security_logs",
    user="analyst",
    password="security123"
)

cursor = conn.cursor()

# Количество событий
cursor.execute("SELECT COUNT(*) FROM auth_events")
result = cursor.fetchone()
print(f"Всего событий: {result[0] if result else 0}")

# Статистика по типам
cursor.execute("""
    SELECT event_type, COUNT(*) 
    FROM auth_events 
    GROUP BY event_type
""")
print("\nПо типам:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]}")

cursor.close()
conn.close()

### 3.5 Очистка перед заданием

```bash
docker stop test-postgres
docker rm test-postgres
```

Теперь вы понимаете, как работает генератор. Переходите к заданиям в README.md — вам нужно упаковать его в Docker и создать docker-compose.yml для всей инфраструктуры.

## Итоги

В этом блоке вы узнали:

1. **Основы Docker**: запуск контейнеров, управление, тома, порты
2. **Dockerfile**: создание собственных образов, слои, кэширование
3. **uv в Docker**: современный подход к управлению зависимостями

В заданиях вы примените эти знания:
- Напишете Dockerfile для реального приложения
- Создадите docker-compose.yml для многоконтейнерной инфраструктуры
- Получите работающую систему генерации данных для следующих блоков курса